# Batch normalization extracted from the conversation

This notebook is a standalone version of the normalization explanation from `conv_dump.md`. The conversation used the CNN sequence:

`Convolution -> Batch normalization -> ReLU`

The goal is to make the Batch Normalization step concrete with small NumPy and PyTorch examples.

## Setup

The examples use only NumPy, PyTorch, and Matplotlib, which are already listed in this repo's `pixi.toml`.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(42)
np.random.seed(42)

## 1. Core idea

A convolution layer produces feature values. Those values can have different scales in different channels or batches. Batch normalization first standardizes activations:

`normalized = (x - mean) / sqrt(variance + epsilon)`

Then it applies two learned parameters:

`output = gamma * normalized + beta`

`gamma` is a learned scale and `beta` is a learned shift. So batch normalization stabilizes values, but it still lets the network learn the final scale and offset it wants.

In [2]:
x = np.array([10.0, 12.0, 14.0, 16.0])

mean = np.mean(x)
variance = np.var(x)
epsilon = 1e-5

x_norm = (x - mean) / np.sqrt(variance + epsilon)

print("original:", x)
print("mean:", mean)
print("variance:", variance)
print("normalized:", x_norm)
print("new mean:", np.mean(x_norm))
print("new std:", np.std(x_norm))

original: [10. 12. 14. 16.]
mean: 13.0
variance: 5.0
normalized: [-1.342 -0.447  0.447  1.342]
new mean: 0.0
new std: 0.9999990000015


After normalization, the values are centered near 0 and have standard deviation near 1. The small `epsilon` prevents division by zero.

In [3]:
gamma = 2.0  # learned scale
beta = 5.0   # learned shift

y = gamma * x_norm + beta

print("normalized:", x_norm)
print("after learned scale and shift:", y)
print("new mean:", np.mean(y))
print("new std:", np.std(y))

normalized: [-1.342 -0.447  0.447  1.342]
after learned scale and shift: [2.317 4.106 5.894 7.683]
new mean: 5.0
new std: 1.999998000003


## 2. BatchNorm2d normalizes per channel

For CNNs, tensors usually have this shape:

`batch x channels x height x width`

`BatchNorm2d(num_features=C)` normalizes each of the `C` channels separately. For each channel it computes the mean and variance over the batch dimension and the spatial dimensions.

In [4]:
# batch=2, channels=3, height=4, width=4
x = np.random.randn(2, 3, 4, 4)

# Mean and variance are computed per channel.
# axis 0 = batch, axis 2 = height, axis 3 = width
mean = x.mean(axis=(0, 2, 3), keepdims=True)
var = x.var(axis=(0, 2, 3), keepdims=True)

x_norm = (x - mean) / np.sqrt(var + 1e-5)

print("input shape:", x.shape)
print("mean shape:", mean.shape)
print("normalized shape:", x_norm.shape)
print("per-channel normalized means:", x_norm.mean(axis=(0, 2, 3)))
print("per-channel normalized stds:", x_norm.std(axis=(0, 2, 3)))

input shape: (2, 3, 4, 4)
mean shape: (1, 3, 1, 1)
normalized shape: (2, 3, 4, 4)
per-channel normalized means: [-0.  0. -0.]
per-channel normalized stds: [1. 1. 1.]


The important detail is `axis=(0, 2, 3)`: normalize across batch, height, and width, while keeping channels separate.

In [5]:
batch_norm = nn.BatchNorm2d(num_features=3)

x_torch = torch.randn(2, 3, 4, 4)
y_torch = batch_norm(x_torch)

print("input shape:", tuple(x_torch.shape))
print("output shape:", tuple(y_torch.shape))
print("per-channel output means:", y_torch.mean(dim=(0, 2, 3)).detach().numpy())
print("per-channel output stds:", y_torch.std(dim=(0, 2, 3), unbiased=False).detach().numpy())

input shape: (2, 3, 4, 4)
output shape: (2, 3, 4, 4)
per-channel output means: [0. 0. 0.]
per-channel output stds: [1. 1. 1.]


## 3. Where it sits in a CNN block

A common block is:

`Conv2d -> BatchNorm2d -> ReLU`

If the convolution outputs 8 channels, then `BatchNorm2d` also needs 8 features.

In [6]:
block = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, padding=1),
    nn.BatchNorm2d(num_features=8),
    nn.ReLU(),
)

image_batch = torch.randn(4, 3, 32, 32)
features = block(image_batch)

print("input:", tuple(image_batch.shape))
print("output:", tuple(features.shape))

input: (4, 3, 32, 32)
output: (4, 8, 32, 32)


## 4. Training mode vs inference mode

During training, batch normalization looks at the current mini-batch and updates its stored running statistics. Those values are kept inside the BatchNorm layer itself, not recalculated from scratch every time.

During inference, the layer switches to those stored values instead of the new batch. The next cell shows the idea with plain Python numbers first, so the behavior is easier to see before jumping back to PyTorch.

In [13]:
# A tiny BatchNorm-like toy example with plain Python numbers.
# The same idea applies to each channel in BatchNorm2d.

train_batch_1 = [8.0, 10.0, 12.0, 14.0]
train_batch_2 = [9.0, 11.0, 13.0, 15.0]

# During training, BatchNorm would look at the batch and update its running stats.
running_mean = sum(train_batch_1) / len(train_batch_1)
running_var = sum((value - running_mean) ** 2 for value in train_batch_1) / len(train_batch_1)

print("running mean learned during training:", round(running_mean, 2))
print("running var learned during training:", round(running_var, 2))

# In inference mode, we do NOT recompute mean and variance from the new batch.
# We reuse the stored training values inside the layer.
new_input = [20.0]
inference_output = (new_input[0] - running_mean) / (running_var + 1e-5) ** 0.5

print("new input during inference:", new_input[0])
print("normalized with stored training stats:", round(inference_output, 2))

running mean learned during training: 11.0
running var learned during training: 5.0
new input during inference: 20.0
normalized with stored training stats: 4.02


This is why inference code should normally call `model.eval()`. If a model stays in training mode during inference, batch normalization will use the current batch instead of the learned running statistics.

## 5. Small batch caveat

Batch normalization can be noisy with very small batches, especially batch size 1. For CNNs, `BatchNorm2d` can still average over height and width, but the statistics are less stable than with larger batches.

Common alternatives for very small batches include:

- `GroupNorm`
- `LayerNorm`
- `InstanceNorm`

In [8]:
group_norm = nn.GroupNorm(num_groups=4, num_channels=8)

small_batch_features = torch.randn(1, 8, 16, 16)
normalized = group_norm(small_batch_features)

print("input shape:", tuple(small_batch_features.shape))
print("output shape:", tuple(normalized.shape))

input shape: (1, 8, 16, 16)
output shape: (1, 8, 16, 16)


## Mental model

Batch normalization says: these feature values are unstable, so I will center and scale them, then let the network learn the best final scale and shift.

In the CNN diagram:

`Convolution` creates feature maps.

`Batch normalization` stabilizes feature-map values channel by channel.

`ReLU` keeps positive signals and removes negative ones.